# PA3 Task 3 — Kaggle runner for the live experiment (Step 4)

Only Step 4 needs a GPU: 50 fixed MATH questions × {single 3,072 | parallel 3 × 1,024 | refine 1,024 + 2,048}
with `Qwen/Qwen2.5-3B-Instruct` (pinned revision), FP16, batch size 1. All calls run on **one** GPU so runtimes are
comparable. The run is resumable: every finished call is appended to `live_ckpt/live_calls.jsonl`.

Everything offline (Steps 1–3, 5, 6) runs on a CPU from the supplied export (`python scripts/run_task3.py --step all`).
This notebook needs only the repository: the 50 questions are in `data/ids/live_questions.json`.

Notebook settings: **Accelerator = GPU T4 ×1 (or ×2; only GPU 0 is used)**, **Internet = On**.
Run with *Save Version → Save & Run All (Commit)*; expected ~1.5–3 h.

In [ ]:
# ---------------- settings (edit these) ----------------
REPO_URL      = "https://github.com/wasaymj/ATML.git"
BRANCH        = "main"
PREV_OUTPUT   = ""      # e.g. "/kaggle/input/pa3-task3-live" (Add Input -> Your Work -> this notebook) to resume
SMOKE_TEST    = False   # True: only the first 2 questions (check the setup, ~5 min)

import os, subprocess, sys, shutil, glob
WORK = "/kaggle/working"
REPO = f"{WORK}/ATML"
TASK = f"{REPO}/PA3/Task 3"
CKPT = f"{WORK}/live_ckpt"
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
# ---------------- get the code ----------------
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
assert os.path.exists(f"{TASK}/data/ids/live_questions.json"), "run `--step prepare` locally and push data/ids first"

In [ ]:
# ---------------- dependencies (torch stays Kaggle's CUDA build) ----------------
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{TASK}/requirements.txt"])
print("pip returncode", r.returncode)
print(subprocess.run([sys.executable, "-c", "import torch, transformers, math_verify; print('torch', torch.__version__, "
      "'| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())"], capture_output=True, text=True).stdout)

In [ ]:
# ---------------- resume from a previous version ----------------
if PREV_OUTPUT and os.path.exists(f"{PREV_OUTPUT}/live_ckpt"):
    shutil.copytree(f"{PREV_OUTPUT}/live_ckpt", CKPT, dirs_exist_ok=True)
    print("restored", sum(1 for _ in open(f"{CKPT}/live_calls.jsonl")), "finished calls")

In [ ]:
# ---------------- live generation (GPU 0 only) ----------------
env = dict(os.environ, CUDA_VISIBLE_DEVICES="0", TOKENIZERS_PARALLELISM="false")
cmd = [sys.executable, "scripts/run_task3.py", "--step", "live", "--live_ckpt", CKPT]
if SMOKE_TEST:
    cmd += ["--max_questions", "2"]
print("$", " ".join(cmd), flush=True)
rc = subprocess.run(cmd, cwd=TASK, env=env).returncode
print("returncode", rc)

In [ ]:
# ---------------- summary + package the outputs ----------------
import json
s = json.load(open(f"{TASK}/results/live/live_summary.json"))
for k, v in s["strategies"].items():
    print(f"{k:9s} acc {100*v['acc'][0]:5.1f}  out tok {v['mean_output_tokens']:7.1f}  time {v['mean_time_s']:6.1f} s  "
          f"trunc {100*v['truncation_rate_calls']:4.1f}%  no-box {100*v['no_extraction_rate_calls']:4.1f}%")
print("revision transitions", s["revision_transitions"])
shutil.make_archive(f"{WORK}/task3_live_results", "zip", root_dir=TASK, base_dir="results/live")
shutil.make_archive(f"{WORK}/task3_live_ckpt", "zip", root_dir=WORK, base_dir="live_ckpt")
print(os.listdir(WORK))